# Grasshopper Optimization for Energy Load Balancing in Smart Grid

**Workflow**
1. Install dependencies & clear cache
2. Imports & setup (autoreload ON)
3. Load dataset
4. Preprocess data — 10 enhanced features
5. Train & compare models: Random Forest, SVR, XGBoost
6. Model comparison table + bar chart
7. Evaluate best model (RMSE / MAE / R²)
8. Apply GOA optimisation (best model predictions)
9. Evaluate grid KPIs (Before vs After)
10. Visualise results

## Step 0 – Install Dependencies & Clear Cache

> **Why clear cache?**  
> Python caches compiled `.pyc` files in `__pycache__/`. If `src/` files were
> updated after the kernel last imported them, the kernel keeps using the **old**
> compiled version. Deleting the cache forces a fresh compile.
>
> After running this cell, **RESTART THE KERNEL once**, then run all cells.

In [ ]:
import subprocess, sys, shutil, os

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'pandas', 'numpy', 'matplotlib', 'scikit-learn', 'joblib', 'xgboost',
    '--quiet'
])
print('Packages installed.')

PROJECT_ROOT_TEMP = os.path.abspath(os.path.join(os.getcwd(), '..'))
cache_dir = os.path.join(PROJECT_ROOT_TEMP, 'src', '__pycache__')
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f'Cleared cache -> {cache_dir}')
else:
    print('No cache to clear.')

print('\nStep 0 complete. Now RESTART THE KERNEL once, then run all cells.')

## Step 1 – Imports & Setup

> `%autoreload 2` automatically reloads any changed `src/` module before every
> cell execution — no kernel restart needed after editing source files.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Project root :', PROJECT_ROOT)
print('Results dir  :', RESULTS_DIR)
print('autoreload   : ON')
print('Environment ready.')

## Step 2 – Load Dataset

In [ ]:
DATASET_PATH = os.path.join(PROJECT_ROOT, 'dataset', 'DUQ_hourly.csv')

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f'Dataset not found: {DATASET_PATH}')

print(f'Dataset found -> {DATASET_PATH}')
preview = pd.read_csv(DATASET_PATH, nrows=3)
print(f'Columns: {list(preview.columns)}')
preview

## Step 3 – Preprocess Data (10 Enhanced Features)

| Feature | Type | Why it helps |
|---------|------|--------------|
| `hour_sin`, `hour_cos` | Cyclical | Removes 23->0 discontinuity |
| `day_of_week` | Ordinal | Weekday vs weekend pattern |
| `month` | Ordinal | Seasonal variation |
| `is_weekend` | Binary | Lower load on weekends |
| `tou_price` | Continuous | Time-of-use price proxy |
| `lag_1`, `lag_2`, `lag_3` | **Lag** | **Biggest R² driver** — load at t ≈ load at t-1 |
| `rolling_mean_24` | Rolling | Daily baseline level |

In [ ]:
from src.preprocessing import preprocess, FEATURE_COLS

X, y, scaler, raw_df = preprocess(DATASET_PATH)

expected_n_features = len(FEATURE_COLS)

print(f'\nFeature matrix : {X.shape}   <- expected (N, {expected_n_features})')
print(f'Target vector  : {y.shape}')
print(f'Features       : {FEATURE_COLS}')

assert X.shape[1] == expected_n_features, (
    f'ERROR: got {X.shape[1]} features instead of {expected_n_features}. '
    'Check preprocess() output and FEATURE_COLS alignment.'
)
print(f'\nCorrect — {expected_n_features} features confirmed. Proceeding...')
raw_df.head()

## Step 4 – Train & Compare Models (RF, SVR, XGBoost)

All three models are trained on the same chronological split.  
The **best model** (highest R²) is automatically selected for GOA.

| Model | Notes |
|-------|-------|
| Random Forest | RandomizedSearchCV (10 iter × 3-fold) when `use_search=True` |
| SVR | RBF kernel; features scaled with StandardScaler |
| XGBoost | 300 trees, lr=0.05, depth=6 |

In [ ]:
from src.forecasting_model import run_forecasting_pipeline

# use_search=True  -> RandomizedSearchCV for RF (~2-3 min)
# use_search=False -> fixed params (fast demo)
best_model, best_metrics, y_test, y_pred, X_test, all_metrics, all_preds = \
    run_forecasting_pipeline(X, y, use_search=False)

print('\nBest model metrics:', best_metrics)

## Step 5 – Model Comparison Bar Chart (R² and RMSE)

In [ ]:
model_names = list(all_metrics.keys())
r2_values   = [all_metrics[m]['R2']   for m in model_names]
rmse_values = [all_metrics[m]['RMSE'] for m in model_names]

colors = ['steelblue', 'tomato', 'seagreen']
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

bars = axes[0].bar(model_names, r2_values, color=colors, alpha=0.85, width=0.4)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.005,
                 f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)
axes[0].set_title('Model Comparison — R²')
axes[0].set_ylabel('R²')
axes[0].grid(axis='y', alpha=0.3)

bars = axes[1].bar(model_names, rmse_values, color=colors, alpha=0.85, width=0.4)
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.005,
                 f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
axes[1].set_title('Model Comparison — RMSE')
axes[1].set_ylabel('RMSE (kWh)')
axes[1].grid(axis='y', alpha=0.3)

fig.suptitle('ML Model Comparison', fontsize=12)
plt.tight_layout()
out = os.path.join(RESULTS_DIR, 'model_comparison.png')
plt.savefig(out, dpi=150)
print(f'Saved -> {out}')
plt.show()

## Step 6 – Best Model: Actual vs Predicted

> | Metric | Meaning | Ideal |
> |--------|---------|-------|
> | **RMSE** | Avg error in kWh — penalises large deviations | -> 0 |
> | **MAE** | Plain avg absolute error in kWh | -> 0 |
> | **R²** | % of load variance explained by the model | -> 1.0 (target >= 0.95) |

In [ ]:
from src.evaluation import evaluate_model_performance

ml_metrics = evaluate_model_performance(y_test, y_pred)
print('\nReturned metrics dict:', ml_metrics)

## Step 7 – GOA Optimisation (Best Model Predictions)

In [ ]:
from src.goa_optimization import grasshopper_optimization

test_start = len(y) - len(y_test)
price_test = raw_df['tou_price'].values[test_start : test_start + len(y_test)]

goa_result = grasshopper_optimization(
    predicted_load=y_pred, price=price_test,
    n_grasshoppers=30, max_iter=100,
)
optimized_load = goa_result['optimized_load']
print('GOA complete. Best fitness:', goa_result['best_fitness'])

## Step 8 – Grid KPIs: Before vs After GOA

In [ ]:
from src.evaluation import compare_before_after

kpi_df = compare_before_after(y_pred, optimized_load, price_test)
kpi_df

## Step 9 – Visualisations

In [ ]:
# --- Before vs After load curve ---
plt.figure(figsize=(12, 5))
plt.plot(y_pred,         label='Before GOA', color='tomato',   linewidth=1.4)
plt.plot(optimized_load, label='After GOA',  color='seagreen', linewidth=1.4)
plt.axhline(y_pred.max(),        color='red',       linestyle=':', linewidth=1,
            label=f'Peak before = {y_pred.max():.1f} kWh')
plt.axhline(optimized_load.max(), color='darkgreen', linestyle=':', linewidth=1,
            label=f'Peak after  = {optimized_load.max():.1f} kWh')
plt.title('Load Schedule: Before vs After GOA Optimisation')
plt.xlabel('Time Step (hours)')
plt.ylabel('Load (kWh)')
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
out = os.path.join(RESULTS_DIR, 'before_after_load.png')
plt.savefig(out, dpi=150)
print(f'Saved -> {out}')
plt.show()

# --- Cost comparison ---
before_cost = float(np.sum(y_pred * price_test))
after_cost  = float(np.sum(optimized_load * price_test))
plt.figure(figsize=(6, 4))
bars = plt.bar(['Before GOA', 'After GOA'], [before_cost, after_cost],
               color=['tomato', 'seagreen'], alpha=0.85, width=0.4)
for bar in bars:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
             f'${bar.get_height():,.1f}', ha='center', va='bottom', fontsize=9)
plt.title('Total Electricity Cost: Before vs After GOA')
plt.ylabel('Total Cost ($)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
out = os.path.join(RESULTS_DIR, 'cost_comparison.png')
plt.savefig(out, dpi=150)
print(f'Saved -> {out}')
plt.show()

# --- GOA convergence ---
plt.figure(figsize=(8, 4))
plt.plot(goa_result['fitness_history'], color='purple', linewidth=1.5)
plt.title('GOA Convergence — Best Fitness over Iterations')
plt.xlabel('Iteration')
plt.ylabel('Best Fitness')
plt.grid(alpha=0.3)
plt.tight_layout()
out = os.path.join(RESULTS_DIR, 'goa_convergence.png')
plt.savefig(out, dpi=150)
print(f'Saved -> {out}')
plt.show()

print('\nAll visualisations saved to results/')